# 02｜臺北市 YouBike

1.	從新雲端學院下載 02_讀程式練習.ipynb，上傳到雲端硬碟的 `115-1_GISprog/notebooks/`。
2.	在 Colab 開啟並執行一遍，完成裡面標示 【練習】 的儲存格。
3.	另外建立 `notebooks/02_youbike_json.ipynb`，這份才是要繳交的作業。
4.	第一個儲存格沿用 00_Colab環境設定範本.`ipynb` 的路徑設定。


## 1. 路徑設定

沿用 `00_Colab環境設定範本.ipynb` 的路徑設定：在 Colab 掛載 Google Drive，本機執行時則以 notebook 上一層作為專案資料夾。之後一律用 `RAW_DIR`、`PROCESSED_DIR`、`OUTPUT_DIR` 組合路徑。

In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/115-1_GISprog")
else:
    # 備援：本機執行時，notebook 位於 115-1_GISprog/notebooks/，上一層就是專案資料夾
    PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FONT_DIR = PROJECT_DIR / "fonts"

if not RAW_DIR.exists():
    raise FileNotFoundError(f"找不到 {RAW_DIR}，請確認 115-1_GISprog 資料夾位置是否正確")

print("執行環境：", "Google Colab" if IN_COLAB else "本機")
print("專案資料夾：", PROJECT_DIR)
print("原始資料：", sorted(p.name for p in RAW_DIR.iterdir()))

執行環境： 本機
專案資料夾： c:\作業資料夾\112-1~115-1 程式設計相關課程\課堂習題或回家作業\python\python\GIS_Programming
原始資料： ['worldcities.csv', '資料來源說明.md']


## 2. 檢查資料

取得臺北市 YouBike 2.0 即時資料：
https://tcgbusfs.blob.core.windows.net/dotapp/youbike/v2/youbike_immediate.json
印出並在筆記中記錄：
* 資料的型態（list 還是 dict？）
* 總筆數
* 第一筆的所有 key，以及每個 key 的值是什麼型態

In [10]:
import requests

URL = "https://tcgbusfs.blob.core.windows.net/dotapp/youbike/v2/youbike_immediate.json"

response = requests.get(URL, timeout=30)
response.raise_for_status()  # 若下載失敗（狀態碼不是 2xx）就直接報錯

data = response.json()
print("HTTP 狀態碼：", response.status_code)
print("資料型態：", type(data).__name__)
print("總筆數：", len(data))
print("-----------------------------------")
print("第一筆資料內容")

first = data[0]

for key, value in first.items():
    print(f"{key:24} {type(value).__name__:6} {value!r}")

HTTP 狀態碼： 200
資料型態： list
總筆數： 1803
-----------------------------------
第一筆資料內容
sno                      str    '500101001'
sna                      str    'YouBike2.0_捷運科技大樓站'
sarea                    str    '大安區'
mday                     str    '2026-09-24 11:49:02'
ar                       str    '復興南路二段235號前'
sareaen                  str    'Daan Dist.'
snaen                    str    'YouBike2.0_MRT Technology Bldg. Sta.'
aren                     str    'No.235， Sec. 2， Fuxing S. Rd.'
act                      str    '1'
srcUpdateTime            str    '2026-09-24 11:49:52'
updateTime               str    '2026-09-24 11:49:52'
infoTime                 str    '2026-09-24 11:49:02'
infoDate                 str    '2026-09-24'
Quantity                 int    28
available_rent_bikes     int    3
latitude                 float  25.02605
longitude                float  121.5436
available_return_bikes   int    23


## 3. 寫函式 `fetch_youbike()`

要求：
* 有 timeout 參數
* 有錯誤處理：網路失敗或對方回傳錯誤時，要讓你知道失敗了
* 回傳 list of dict
> **注意：** except: pass 這種寫法會把錯誤「吞掉」，讓你以為成功了。這是決策點 5。


In [11]:
import requests

YOUBIKE_URL = "https://tcgbusfs.blob.core.windows.net/dotapp/youbike/v2/youbike_immediate.json"


def fetch_youbike(timeout=30):
    """抓取臺北市 YouBike 2.0 即時資料，成功時回傳 list of dict。

    timeout：最多等待幾秒。
    失敗時不會回傳空資料，而是丟出 RuntimeError 並說明原因。
    """
    try:
        response = requests.get(YOUBIKE_URL, timeout=timeout)
        response.raise_for_status()  # 狀態碼不是 2xx（例如 404、500）就丟出 HTTPError
        data = response.json()
    except requests.exceptions.Timeout as e:
        raise RuntimeError(f"連線逾時：{timeout} 秒內沒有收到回應") from e
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"伺服器回傳錯誤：HTTP {response.status_code}") from e
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"網路連線失敗：{e}") from e
    except ValueError as e:
        raise RuntimeError("回傳的內容不是有效的 JSON") from e

    # 確認格式符合預期：最外層是 list，裡面每一筆是 dict
    if not isinstance(data, list):
        raise RuntimeError(f"資料格式不符：預期 list，實際是 {type(data).__name__}")

    return data

In [12]:
# 測試：正常呼叫
stations = fetch_youbike(timeout=10)
print("型態：", type(stations).__name__)
print("筆數：", len(stations))

型態： list
筆數： 1803


## 4.寫函式 `summarize_by_area(records)`
用 `for` 迴圈與 `dict`（不要用 `Pandas`），回傳每個 `sarea` 的三個數字：
| 統計項目 | 說明 |
|:---:|:---:|
| 站數 | 這一區總共有幾站 |
| 營運中站數 | 只算營運中的站 |
| 可借車總數 | 由你決定：要不要包含暫停營運的站？在筆記說明理由 |

回傳格式自己決定（例如 dict of dict），但要在 docstring 寫清楚。


In [13]:
def summarize_by_area(records):
    """依行政區（sarea）統計 YouBike 站點。

    參數：
        records：list of dict，每個 dict 是一個站點（fetch_youbike() 的回傳值）。

    回傳：
        dict of dict，外層 key 是行政區名稱，內層是三個統計數字：
        {
            "大安區": {"站數": 100, "營運中站數": 98, "可借車總數": 850},
            ...
        }
        - 站數：該區所有站點，不論是否營運。
        - 營運中站數：act 為 "1" 的站點數。
        - 可借車總數：只加總營運中站點的 available_rent_bikes。
    """
    summary = {}

    for station in records:
        area = station["sarea"]

        # 第一次遇到這個區，先建立空白的統計
        if area not in summary:
            summary[area] = {"站數": 0, "營運中站數": 0, "可借車總數": 0}

        summary[area]["站數"] += 1

        if station["act"] == "1":
            summary[area]["營運中站數"] += 1
            summary[area]["可借車總數"] += station["available_rent_bikes"]

    return summary

In [14]:
# 測試：依行政區統計並印出結果
area_summary = summarize_by_area(stations)

for area, numbers in area_summary.items():
    print(area, numbers)

大安區 {'站數': 216, '營運中站數': 212, '可借車總數': 1292}
大同區 {'站數': 84, '營運中站數': 84, '可借車總數': 751}
士林區 {'站數': 155, '營運中站數': 151, '可借車總數': 1370}
文山區 {'站數': 124, '營運中站數': 121, '可借車總數': 1084}
中正區 {'站數': 138, '營運中站數': 137, '可借車總數': 1158}
中山區 {'站數': 205, '營運中站數': 198, '可借車總數': 1688}
內湖區 {'站數': 226, '營運中站數': 224, '可借車總數': 1638}
北投區 {'站數': 133, '營運中站數': 131, '可借車總數': 1305}
松山區 {'站數': 108, '營運中站數': 107, '可借車總數': 968}
南港區 {'站數': 122, '營運中站數': 122, '可借車總數': 1020}
信義區 {'站數': 129, '營運中站數': 129, '可借車總數': 1219}
萬華區 {'站數': 98, '營運中站數': 92, '可借車總數': 781}
臺大公館校區 {'站數': 65, '營運中站數': 65, '可借車總數': 833}


In [16]:
# 檢查：暫停營運的站，可借車數是不是 0？
suspended_count = 0
suspended_with_bikes = 0
suspended_bikes_total = 0

for station in stations:
    if station["act"] != "1":
        suspended_count += 1
        suspended_bikes_total += station["available_rent_bikes"]
        if station["available_rent_bikes"] > 0:
            suspended_with_bikes += 1

print("暫停營運的站數：", suspended_count)
print("其中可借車數大於 0 的站數：", suspended_with_bikes)
print("暫停營運站的可借車總數：", suspended_bikes_total)

暫停營運的站數： 30
其中可借車數大於 0 的站數： 1
暫停營運站的可借車總數： 4


## 5. 寫 `assert` 測試
用你自己手造的 3–5 筆資料（不要用真實的 1,800 筆）寫至少 3 個 `assert`：
* 至少一個要測暫停營運的站有沒有被正確處理
* 測試要能真的抓到錯：試著故意把函式改錯，確認你的 `assert` 會失敗


In [20]:
# 手造五筆測試資料（只放 summarize_by_area 會用到的欄位）
# A 區：2 站營運中（可借 3、5）＋ 1 站暫停營運（故意設成可借 4）
# B 區：1 站營運中（可借 0）＋ 1 站暫停營運（可借 0）
test_records = [
    {"sarea": "A區", "act": "1", "available_rent_bikes": 3},
    {"sarea": "A區", "act": "1", "available_rent_bikes": 5},
    {"sarea": "A區", "act": "0", "available_rent_bikes": 4},
    {"sarea": "B區", "act": "1", "available_rent_bikes": 0},
    {"sarea": "B區", "act": "0", "available_rent_bikes": 0},
]

result = summarize_by_area(test_records)

# 1. 兩個區都要出現
assert set(result.keys()) == {"A區", "B區"}, "行政區清單不對"

# 2. 站數：暫停營運的站也要算進去（A 區 3 站）
assert result["A區"]["站數"] == 3, "A區站數應為 3"

# 3. 營運中站數：不含暫停營運（A 區 2 站）
assert result["A區"]["營運中站數"] == 2, "A區營運中站數應為 2"

# 4. 可借車總數：暫停營運站的 4 輛不能算，所以是 3 + 5 = 8，不是 12
assert result["A區"]["可借車總數"] == 8, "A區可借車總數應為 8（不含暫停營運站）"

# 5. B 區：1 站營運中、可借 0 輛
assert result["B區"]["營運中站數"] == 1, "B區營運中站數應為 1"
assert result["B區"]["可借車總數"] == 0, "B區可借車總數應為 0"



In [18]:
def check_summary(func):
    """用 test_records 檢查統計函式，任何一個 assert 不成立就會丟出 AssertionError。"""
    result = func(test_records)

    # 1. 兩個區都要出現
    assert set(result.keys()) == {"A區", "B區"}, "行政區清單不對"

    # 2. 站數：暫停營運的站也要算進去（A 區 3 站）
    assert result["A區"]["站數"] == 3, "A區站數應為 3"

    # 3. 營運中站數：不含暫停營運（A 區 2 站）
    assert result["A區"]["營運中站數"] == 2, "A區營運中站數應為 2"

    # 4. 可借車總數：暫停營運站的 4 輛不能算，所以是 3 + 5 = 8，不是 12
    assert result["A區"]["可借車總數"] == 8, "A區可借車總數應為 8（不含暫停營運站）"

    # 5. B 區：1 站營運中、可借 0 輛
    assert result["B區"]["營運中站數"] == 1, "B區營運中站數應為 1"
    assert result["B區"]["可借車總數"] == 0, "B區可借車總數應為 0"


check_summary(summarize_by_area)
print("所有 assert 通過")

所有 assert 通過


In [21]:
# 故意改錯：把暫停營運站的可借車數也加進去，確認 assert 會失敗
def summarize_by_area_buggy(records):
    summary = {}

    for station in records:
        area = station["sarea"]

        if area not in summary:
            summary[area] = {"站數": 0, "營運中站數": 0, "可借車總數": 0}

        summary[area]["站數"] += 1
        summary[area]["可借車總數"] += station["available_rent_bikes"]  # 錯：沒有排除暫停營運

        if station["act"] == "1":
            summary[area]["營運中站數"] += 1

    return summary


# 這一行預期會出現 AssertionError：A區可借車總數應為 8（不含暫停營運站）
check_summary(summarize_by_area_buggy)

AssertionError: A區可借車總數應為 8（不含暫停營運站）

## 6. 存快照
把原始 JSON 存成檔名含取得時間的檔案，例如 youbike_20260925_0830.json，放在 data/processed/


In [ ]:
import json
from datetime import datetime, timedelta, timezone

# 台灣時間（UTC+8）。Colab 預設是 UTC，直接用 datetime.now() 會差 8 小時
TAIWAN_TZ = timezone(timedelta(hours=8))


def save_youbike_snapshot(records, fetched_at, out_dir):
    """把 YouBike 資料存成檔名含取得時間的 JSON 檔，回傳檔案路徑。

    records：fetch_youbike() 的回傳值（list of dict）。
    fetched_at：取得資料的時間（datetime），檔名格式為 youbike_YYYYMMDD_HHMM.json。
    out_dir：存放的資料夾（Path），不存在會自動建立。
    同一分鐘內重複執行，會覆蓋同名檔案。
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"youbike_{fetched_at:%Y%m%d_%H%M}.json"

    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False)  # 保留中文，不轉成 \uXXXX

    return path

# 重新抓一次資料，並以「抓到資料的時間」命名存檔
snapshot = fetch_youbike(timeout=30)
fetched_at = datetime.now(TAIWAN_TZ)

snapshot_path = save_youbike_snapshot(snapshot, fetched_at, PROCESSED_DIR)

print("已儲存：", snapshot_path)
print("筆數：", len(snapshot))
print("檔案大小（KB）：", round(snapshot_path.stat().st_size / 1024, 1))

已儲存： c:\作業資料夾\112-1~115-1 程式設計相關課程\課堂習題或回家作業\python\python\GIS_Programming\data\processed\youbike_20260924_1547.json
筆數： 1803
檔案大小（KB）： 985.7
